# Content Engine — Free Google Colab Workflow

This notebook turns any YouTube video you own or are authorized to process into **three real, downloadable vertical MP4 clips** (20s, 30s, 35s) at 1080×1920 with burned-in captions.

**No payment card, no API keys, no server, no local install.** It reuses the project's real acquisition, transcription, clip-selection, caption, and rendering code.

### How to use
1. Run each cell top-to-bottom by pressing **Shift+Enter**.
2. When a cell shows a form field, type your YouTube URL and press Enter.
3. Wait for each cell to finish (a number appears in `In [n]:`) before running the next.

### Important warnings
- **Colab sessions disconnect** after ~90 minutes of inactivity and temporary files are lost. Download your clips before closing the tab.
- **Free tier limits**: CPU runtimes are ~12h, GPU runtimes are shorter. Rendering uses CPU (no GPU needed).
- **Only process videos you own or are authorized to use.**

## Cell 1 — Install dependencies

This installs FFmpeg (for video rendering + probing), yt-dlp (for downloads), faster-whisper (for transcription, free and local), and downloads the project code from the local upload.

In [ ]:
# Install FFmpeg and ffprobe (Colab already has apt available)
import subprocess, sys, shutil

def run(cmd, check=True):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if check and result.returncode != 0:
        print("STDERR:", result.stderr[-500:])
        raise RuntimeError(f"Command failed: {' '.join(cmd[:4])}...")
    return result

# FFmpeg is usually pre-installed on Colab; verify and install if missing
if shutil.which("ffmpeg") is None:
    print("FFmpeg not found, installing via apt...")
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg"])

# Verify ffmpeg + ffprobe
ff = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
fp = subprocess.run(["ffprobe", "-version"], capture_output=True, text=True)
if ff.returncode != 0:
    raise RuntimeError("FFmpeg is not working. This Colab runtime may not support apt installs.")
if fp.returncode != 0:
    raise RuntimeError("ffprobe is not working. FFmpeg installation appears incomplete.")

print("FFmpeg:", ff.stdout.split(chr(10))[0])
print("ffprobe:", fp.stdout.split(chr(10))[0])

# Verify libx264 encoder and ass filter (required for rendering + captions)
enc = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True)
fil = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True)
if "libx264" not in enc.stdout:
    raise RuntimeError("libx264 encoder is missing — rendering cannot produce H.264 video.")
if "ass" not in fil.stdout:
    raise RuntimeError("libass subtitle filter is missing — captions cannot be burned in.")
print("libx264 encoder: OK")
print("ass subtitle filter: OK")

# Install Python packages
print("\nInstalling Python packages...")
run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp", "faster-whisper"])
print("yt-dlp + faster-whisper installed.")

## Cell 2 — Upload project code

The notebook reuses the project's real Python modules. Upload the `backend/` folder from the Content Engine project (or it will be fetched if you cloned the repo to Colab).

**Option A (easiest):** If you have the project as a ZIP, upload it when prompted.

**Option B:** If the project is on your Google Drive, skip to the Drive cell and mount first.

In [ ]:
import os, sys, pathlib

PROJECT_DIR = pathlib.Path("/content/content-engine")

if PROJECT_DIR.exists():
    print(f"Project already exists at {PROJECT_DIR}")
else:
    # Try to find backend/ in common locations
    found = False
    for candidate in [
        pathlib.Path("/content/backend"),
        pathlib.Path("/content/content-engine/backend"),
    ]:
        if (candidate / "app" / "providers" / "renderer.py").exists():
            PROJECT_DIR = candidate.parent
            found = True
            print(f"Found project at {PROJECT_DIR}")
            break
    
    if not found:
        print("Project code not found. Please upload the project ZIP or clone it.")
        print("To upload: click the folder icon in the left sidebar, then upload the ZIP.")
        print("Then unzip it and ensure /content/content-engine/backend/app/ exists.")
        print("\nOr set PROJECT_DIR manually below:")
        print("  PROJECT_DIR = pathlib.Path('/path/to/your/project')")
        raise FileNotFoundError("Project code not found. See instructions above.")

# Add backend/ to Python path so 'app.*' imports resolve
BACKEND_DIR = PROJECT_DIR / "backend"
sys.path.insert(0, str(BACKEND_DIR))

# Verify the real modules are importable
from app.providers.renderer import FFmpegRenderer, TARGET_WIDTH, TARGET_HEIGHT
from app.providers.clip_detection import HeuristicClipDetector, TARGET_DURATIONS
from app.providers.captions import TranscriptCaptionProvider
from app.providers.transcription import FasterWhisperProvider
from app.providers.downloader import YtDlpDownloader
from app.providers.video_source import YtDlpSourceProvider
from app.providers.media_probe import get_media_probe
from app.storage.local import LocalStorage
from app.core.config import Settings

print(f"Project loaded from: {PROJECT_DIR}")
print(f"Backend path: {BACKEND_DIR}")
print(f"Target clips: {TARGET_DURATIONS} seconds each")
print(f"Output resolution: {TARGET_WIDTH}x{TARGET_HEIGHT} (9:16 portrait)")

## Cell 3 — Enter your YouTube URL

Type or paste the URL of a YouTube video you own or are authorized to process. The video must be at least 40 seconds long (the longest clip is 35s, plus a small margin for clip selection).

In [ ]:
# @title Enter YouTube URL
YOUTUBE_URL = ""  # @param {type:"string"}

if not YOUTUBE_URL.strip():
    YOUTUBE_URL = input("Paste your YouTube URL and press Enter: ")

YOUTUBE_URL = YOUTUBE_URL.strip()
if not YOUTUBE_URL:
    raise ValueError("No URL provided. Please run this cell again and enter a YouTube URL.")

# Basic URL validation
from urllib.parse import urlparse
parsed = urlparse(YOUTUBE_URL)
if parsed.scheme not in ("http", "https"):
    raise ValueError(f"URL must start with http:// or https:// — got '{parsed.scheme}://'")
if not parsed.hostname:
    raise ValueError("URL must include a hostname (e.g. youtube.com or youtu.be)")
if "youtube.com" not in parsed.hostname and "youtu.be" not in parsed.hostname:
    raise ValueError(
        f"URL hostname '{parsed.hostname}' does not look like YouTube. "
        "Only youtube.com or youtu.be URLs are supported."
    )

print(f"URL accepted: {YOUTUBE_URL}")
print("\nNote: Only process videos you own or are authorized to use.")

## Cell 4 — Set up storage and configuration

Creates a working directory for source videos and rendered clips.

In [ ]:
import pathlib, os

WORK_DIR = pathlib.Path("/content/content_engine_work")
STORAGE_DIR = WORK_DIR / "storage"
OUTPUT_DIR = WORK_DIR / "output"
STORAGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Build settings with the real config class — all from code, no .env needed
os.environ["CE_STORAGE_PATH"] = str(STORAGE_DIR)
os.environ["CE_FFMPEG_PATH"] = "ffmpeg"
os.environ["CE_FFPROBE_PATH"] = "ffprobe"
os.environ["CE_WHISPER_MODEL"] = "tiny"  # ~75MB, fits free Colab RAM
os.environ["CE_WHISPER_DEVICE"] = "cpu"
os.environ["CE_WHISPER_COMPUTE_TYPE"] = "int8"
os.environ["CE_MAX_SOURCE_BYTES"] = str(500 * 1024 * 1024)  # 500 MB limit
os.environ["CE_DOWNLOAD_TIMEOUT_SECONDS"] = "300"

# Clear any cached settings from a previous run
from app.core.config import clear_settings_cache
clear_settings_cache()
from app.core.config import get_settings
settings = get_settings()

# Build the real storage provider
storage = LocalStorage(STORAGE_DIR)

print(f"Storage directory: {STORAGE_DIR}")
print(f"Whisper model: {settings.ce_whisper_model} (device={settings.ce_whisper_device})")
print(f"Max source size: {settings.ce_max_source_bytes // (1024*1024)} MB")
print(f"FFmpeg path: {settings.ce_ffmpeg_path}")
print(f"ffprobe path: {settings.ce_ffprobe_path}")

## Cell 5 — Download the source video

Uses the project's real `YtDlpSourceProvider` to download and validate the video. This is the same code the backend worker uses.

In [ ]:
import time, traceback
from app.providers.downloader import YtDlpDownloader
from app.providers.video_source import YtDlpSourceProvider
from app.providers.media_probe import get_media_probe
from app.core.errors import ContentEngineError

downloader = YtDlpDownloader()
probe = get_media_probe("ffprobe")
provider = YtDlpSourceProvider(
    storage=storage,
    downloader=downloader,
    probe=probe,
    settings=settings,
)

job_id = "colab-run"  # used for the storage path

def on_progress(val):
    if val % 10 == 0:
        print(f"  Download progress: {val}%", flush=True)

print(f"Downloading: {YOUTUBE_URL}")
print("This may take a few minutes depending on video length. Please wait...")

try:
    result = provider.acquire(
        YOUTUBE_URL,
        job_id=job_id,
        progress=on_progress,
    )
except ContentEngineError as exc:
    print(f"\nDownload failed: {exc.code} — {exc.message}")
    if exc.code == "source_too_large":
        print("The video exceeds the 500 MB limit. Try a shorter video.")
    elif exc.code == "download_timeout":
        print("The download timed out. Try a shorter video or rerun this cell.")
    elif exc.code == "invalid_source_url":
        print("The URL was rejected. Make sure it's a valid YouTube link.")
    raise
except Exception as exc:
    print(f"\nUnexpected download error: {exc}")
    traceback.print_exc()
    raise

print(f"\nDownload complete!")
print(f"  File: {result.filename}")
print(f"  Size: {result.size_bytes // (1024*1024)} MB")
print(f"  Duration: {result.duration_seconds:.1f}s" if result.duration_seconds else "  Duration: unknown")
print(f"  Resolution: {result.width}x{result.height}" if result.width else "  Resolution: unknown")
print(f"  Stored as: {result.asset_ref}")

if result.duration_seconds and result.duration_seconds < 40:
    raise RuntimeError(
        f"Video is only {result.duration_seconds:.1f}s long. "
        "Need at least 40s to extract three distinct clips (20s + 30s + 35s)."
    )

## Cell 6 — Transcribe the audio

Uses the project's real `FasterWhisperProvider` with the free `tiny` model. This runs entirely on CPU — no API keys, no paid services. The model downloads from HuggingFace on first use (~75 MB, cached for the session).

In [ ]:
from app.providers.transcription import FasterWhisperProvider

transcriber = FasterWhisperProvider(
    storage=storage,
    model_size=settings.ce_whisper_model,
    device=settings.ce_whisper_device,
    compute_type=settings.ce_whisper_compute_type,
)

def on_transcribe_progress(val):
    if val in (10, 50, 100):
        print(f"  Transcription progress: {val}%", flush=True)

print("Loading Whisper model and transcribing audio...")
print("(The 'tiny' model downloads ~75MB on first use, then runs locally on CPU)")
print("This may take several minutes. Please wait...\n")

try:
    transcript = transcriber.transcribe(result.asset_ref, progress=on_transcribe_progress)
except Exception as exc:
    print(f"\nTranscription failed: {exc}")
    print("Possible causes:")
    print("  - The video has no speech (captions require spoken audio)")
    print("  - Insufficient RAM (try a shorter video)")
    print("  - Model download failed (check internet connection)")
    raise

print(f"\nTranscription complete!")
print(f"  Language: {transcript['language']}")
print(f"  Segments: {len(transcript['segments'])}")
print(f"  Duration: {transcript['duration_seconds']:.1f}s")
print(f"  First 3 segments:")
for seg in transcript['segments'][:3]:
    print(f"    [{seg['start']:.1f}-{seg['end']:.1f}s] {seg['text'][:80]}")

## Cell 7 — Select three clip intervals

Uses the project's real `HeuristicClipDetector` to pick three non-overlapping intervals of exactly 20, 30, and 35 seconds covering the most speech-dense segments.

In [ ]:
from app.providers.clip_detection import HeuristicClipDetector

detector = HeuristicClipDetector()

try:
    clips = detector.detect_moments(transcript)
except Exception as exc:
    print(f"Clip selection failed: {exc}")
    print("\nPossible causes:")
    print("  - Video is too short or has too little speech")
    print("  - Transcript has fewer than 3 distinct speech segments")
    raise

print(f"Selected {len(clips)} clips:\n")
for clip in clips:
    print(f"  Clip {clip['index']+1}: {clip['target_duration']}s "
          f"[{clip['start']:.1f}-{clip['end']:.1f}s] — {clip['label']}")
    print(f"    Reason: {clip['reason']}")
    print()

## Cell 8 — Render the three vertical clips with burned-in captions

Uses the project's real `FFmpegRenderer` to produce three MP4 files at 1080×1920, H.264 + AAC, with captions burned in via the ASS subtitle filter. This is the exact same rendering code verified to produce exact-duration clips.

In [ ]:
from app.providers.renderer import FFmpegRenderer
from app.providers.captions import TranscriptCaptionProvider

renderer = FFmpegRenderer(storage=storage, ffmpeg_path="ffmpeg")
captioner = TranscriptCaptionProvider()

rendered_clips = []

print("Rendering clips with FFmpeg (real encoding, not mocked)...\n")

for clip in clips:
    target = int(clip['target_duration'])
    start = float(clip['start'])
    end = float(clip['end'])
    cues = captioner.cues_for_clip(transcript, start, end)
    output_ref = f"clips/{job_id}/clip_{clip['index']}_{target}s.mp4"

    print(f"  Rendering {target}s clip [{start:.1f}-{end:.1f}s] with {len(cues)} caption cue(s)...")

    def on_render_progress(val, t=target):
        if val in (10, 30, 90, 100):
            print(f"    {t}s clip: {val}%", flush=True)

    try:
        stored_ref = renderer.render_vertical(
            source_ref=result.asset_ref,
            start_seconds=start,
            end_seconds=end,
            output_ref=output_ref,
            captions=cues,
            progress=on_render_progress,
        )
    except Exception as exc:
        print(f"\nRendering failed for {target}s clip: {exc}")
        raise

    clip_path = storage.path_for(stored_ref)
    print(f"    Done: {clip_path.name} ({clip_path.stat().st_size // (1024*1024)} MB)")
    rendered_clips.append({
        'target': target,
        'path': str(clip_path),
        'ref': stored_ref,
        'start': start,
        'end': end,
    })

print(f"\nAll {len(rendered_clips)} clips rendered successfully!")

## Cell 9 — Validate outputs with ffprobe

Checks every clip's duration, resolution, video codec, and audio stream. Fails clearly if any clip is missing or invalid.

In [ ]:
import subprocess, json, pathlib

def ffprobe_json(path):
    r = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json",
         "-show_format", "-show_streams", str(path)],
        capture_output=True, text=True, timeout=30, check=True,
    )
    return json.loads(r.stdout)

print("Validating all clips with ffprobe...\n")

all_valid = True
validation_results = []

for clip in rendered_clips:
    path = clip['path']
    target = clip['target']

    if not pathlib.Path(path).is_file():
        print(f"  FAIL: {target}s clip file missing at {path}")
        all_valid = False
        continue

    info = ffprobe_json(path)
    actual_dur = float(info['format']['duration'])
    size = int(info['format']['size'])

    v_stream = next((s for s in info['streams'] if s['codec_type'] == 'video'), None)
    a_stream = next((s for s in info['streams'] if s['codec_type'] == 'audio'), None)

    errors = []
    if abs(actual_dur - target) > 0.15:
        errors.append(f"duration {actual_dur:.3f}s (target {target}s, deviation {abs(actual_dur-target):.3f}s)")
    if v_stream is None:
        errors.append("no video stream")
    else:
        w, h = int(v_stream['width']), int(v_stream['height'])
        if w != 1080 or h != 1920:
            errors.append(f"resolution {w}x{h} (expected 1080x1920)")
        if v_stream['codec_name'] != 'h264':
            errors.append(f"video codec {v_stream['codec_name']} (expected h264)")
    if a_stream is None:
        errors.append("no audio stream")
    elif a_stream['codec_name'] != 'aac':
        errors.append(f"audio codec {a_stream['codec_name']} (expected aac)")

    status = "OK" if not errors else "FAIL"
    if errors:
        all_valid = False

    print(f"  {target}s clip: {status}")
    print(f"    Duration: {actual_dur:.3f}s (target: {target}.000s, dev: {abs(actual_dur-target):.3f}s)")
    print(f"    Resolution: {w}x{h}" if v_stream else "    Resolution: MISSING")
    print(f"    Video: {v_stream['codec_name'] if v_stream else 'MISSING'}")
    print(f"    Audio: {a_stream['codec_name'] if a_stream else 'MISSING'}")
    print(f"    Size: {size:,} bytes ({size//(1024*1024)} MB)")
    if errors:
        for e in errors:
            print(f"    ERROR: {e}")
    print()

    validation_results.append({
        'target': target, 'path': path, 'duration': actual_dur,
        'width': w if v_stream else None, 'height': h if v_stream else None,
        'video_codec': v_stream['codec_name'] if v_stream else None,
        'audio_codec': a_stream['codec_name'] if a_stream else None,
        'size': size, 'valid': not errors,
    })

if not all_valid:
    raise RuntimeError(
        "One or more clips failed validation. See errors above. "
        "The clips may be corrupted or FFmpeg may have produced invalid output."
    )

print("All clips passed validation!")

## Cell 10 — Preview the clips

Displays each rendered clip inline in the notebook so you can watch them before downloading.

In [ ]:
from IPython.display import HTML, display
import html as html_mod

for clip in rendered_clips:
    path = clip['path']
    target = clip['target']
    safe_path = html_mod.escape(path)
    print(f"\n--- {target}s clip preview ---")
    display(HTML(f'''
        <video width="270" height="480" controls>
            <source src="file://{safe_path}" type="video/mp4">
            Your browser does not support the video tag.
        </video>
        <p><small>{html_mod.escape(path)}</small></p>
    '''))

## Cell 11 — Download the clips individually

Downloads each MP4 file to your computer.

In [ ]:
from google.colab import files

for clip in rendered_clips:
    path = clip['path']
    target = clip['target']
    filename = f"content_engine_clip_{target}s.mp4"
    print(f"Downloading {filename}...")
    files.download(path)

print("\nAll clips downloaded to your computer.")

## Cell 12 — Download all clips as a ZIP

Bundles all three clips into one ZIP file for a single download.

In [ ]:
import zipfile, pathlib
from google.colab import files

zip_path = OUTPUT_DIR / "content_engine_clips.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for clip in rendered_clips:
        path = pathlib.Path(clip['path'])
        arcname = f"content_engine_clip_{clip['target']}s.mp4"
        zf.write(path, arcname)
        print(f"  Added: {arcname} ({path.stat().st_size // (1024*1024)} MB)")

print(f"\nZIP created: {zip_path} ({zip_path.stat().st_size // (1024*1024)} MB)")
print("Downloading ZIP...")
files.download(str(zip_path))
print("Done!")

## Cell 13 — (Optional) Copy clips to your Google Drive

This cell is **optional**. It will ask you to authorize Google Drive access. Only run it if you want to save the clips to your Drive for later access.

If you skip this, your clips will be lost when the Colab session disconnects. **Download them first using the cells above.**

In [ ]:
# @title Copy to Google Drive? (optional)
copy_to_drive = False  # @param {type:"boolean"}

if copy_to_drive:
    from google.colab import drive
    drive.mount('/content/drive')

    drive_dir = pathlib.Path('/content/drive/MyDrive/content_engine_clips')
    drive_dir.mkdir(parents=True, exist_ok=True)

    import shutil
    for clip in rendered_clips:
        src = pathlib.Path(clip['path'])
        dst = drive_dir / f"content_engine_clip_{clip['target']}s.mp4"
        shutil.copy2(src, dst)
        print(f"  Copied: {dst} ({dst.stat().st_size // (1024*1024)} MB)")

    print(f"\nAll clips copied to: {drive_dir}")
    print("You can access them from your Google Drive in the 'content_engine_clips' folder.")
else:
    print("Skipped Google Drive copy.")
    print("Remember: download your clips using the cells above before closing this notebook.")
    print("Colab sessions disconnect and temporary files are lost.")

## Done!

You should now have three real vertical MP4 clips:
- **20 seconds** at 1080×1920 with burned-in captions
- **30 seconds** at 1080×1920 with burned-in captions
- **35 seconds** at 1080×1920 with burned-in captions

All clips use H.264 video, AAC audio, and were rendered by the project's real FFmpeg renderer.

### If something went wrong
- **Download failed**: Check the URL is valid and the video is public/unlisted. Some videos restrict downloads.
- **Transcription failed**: The video may have no speech. Captions require spoken audio.
- **Clip selection failed**: The video may be too short (need 40s+) or have too little distinct speech.
- **Rendering failed**: FFmpeg may have run out of memory. Try a shorter or lower-resolution source video.
- **Session disconnected**: Colab free sessions disconnect after inactivity. Rerun all cells from the top.
- **Files disappeared**: Temporary files are lost on disconnect. Download clips immediately after rendering.